In [0]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    roc_auc_score, recall_score, f1_score,
    classification_report, confusion_matrix
)
from pyspark.ml.functions import vector_to_array
from pyspark.sql import functions as F
import mlflow
import mlflow.sklearn

In [0]:
# Read train/test tables (already have features vector + label from MLlib pipeline)
train_spark = spark.read.table("teams.sensorx.train_features_delta")
test_spark = spark.read.table("teams.sensorx.test_features_delta")

# Sample to fit in driver memory (16GB node can't hold 5M+ rows in pandas)
# Stratified sample preserves label distribution
train_sampled = train_spark.sampleBy("label", fractions={0: 0.2, 1: 0.2}, seed=42)
test_sampled = test_spark.sampleBy("label", fractions={0: 0.2, 1: 0.2}, seed=42)

# Write expanded features to parquet (bypasses spark.driver.maxResultSize limit)
tmp_train = "/Volumes/teams/sensorx/data-dump/_temp_train_expanded"
tmp_test = "/Volumes/teams/sensorx/data-dump/_temp_test_expanded"

(train_sampled
    .select(vector_to_array("features").alias("features"), "label")
    .write.mode("overwrite").parquet(tmp_train))

(test_sampled
    .select(vector_to_array("features").alias("features"), "label")
    .write.mode("overwrite").parquet(tmp_test))

# Read directly with pandas (no Spark collect needed)
train_pdf = pd.read_parquet(tmp_train)
test_pdf = pd.read_parquet(tmp_test)

# Expand feature arrays into numpy (float32 to halve memory)
X_train = np.vstack(train_pdf["features"].values).astype(np.float32)
y_train = train_pdf["label"].values

X_test = np.vstack(test_pdf["features"].values).astype(np.float32)
y_test = test_pdf["label"].values

# Free pandas intermediates
del train_pdf, test_pdf

print(f"Train: {X_train.shape[0]:,} rows, {X_train.shape[1]} features")
print(f"Test:  {X_test.shape[0]:,} rows, {X_test.shape[1]} features")
print(f"Train label distribution: {dict(zip(*np.unique(y_train, return_counts=True)))}")
print(f"Test label distribution:  {dict(zip(*np.unique(y_test, return_counts=True)))}")

In [0]:
# Scale features (fit on train only to avoid data leakage)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Feature means (first 6): {X_train_scaled.mean(axis=0)[:6].round(4)}")
print(f"Feature stds  (first 6): {X_train_scaled.std(axis=0)[:6].round(4)}")

In [0]:
# Train MLP — same architecture as MLlib version: 161 -> 16 -> 16 -> 2
mlp = MLPClassifier(
    hidden_layer_sizes=(16, 16),
    activation="relu",
    solver="adam",
    learning_rate_init=0.001,
    max_iter=50,
    batch_size=128,
    random_state=42,
    verbose=True,
)

mlp.fit(X_train_scaled, y_train)
print(f"\nConverged: {mlp.n_iter_} iterations, final loss: {mlp.loss_:.6f}")

In [0]:
# Evaluate on test set
y_pred = mlp.predict(X_test_scaled)
y_prob = mlp.predict_proba(X_test_scaled)[:, 1]

auc = roc_auc_score(y_test, y_prob)
recall_0 = recall_score(y_test, y_pred, pos_label=0)
recall_1 = recall_score(y_test, y_pred, pos_label=1)
f1_0 = f1_score(y_test, y_pred, pos_label=0)
f1_1 = f1_score(y_test, y_pred, pos_label=1)

print(f"AUC:      {auc:.4f}")
print(f"Recall 0: {recall_0:.4f}  |  Recall 1: {recall_1:.4f}")
print(f"F1     0: {f1_0:.4f}  |  F1     1: {f1_1:.4f}")
print("\n" + classification_report(y_test, y_pred, target_names=["No fault", "Fault"]))